# Novel View Synthesis — inference

Runs the chosen pipeline over `dataset/test/`, writes `submission.zip`.

Before running:
1. Attach the contest dataset via `Add Input → Datasets` (note its slug, e.g. `nvs-small`).
2. Set `DATASET_SLUG` below to that slug.
3. Switch the accelerator to GPU (Settings → Accelerator) if the pipeline needs it. The lidar-warp baseline runs on CPU.

In [ ]:
DATASET_SLUG = "nvs-small"        # change to your attached dataset slug
TEST_SUBDIR = "dataset/test"        # path inside the attached dataset
PIPELINE   = "lidar_warp"            # module name under src/pipelines/
REPO_URL    = "https://github.com/Plat1011/Novel-View-Synthesis.git"
REPO_BRANCH = "main"

In [ ]:
import os, subprocess, sys, pathlib

subprocess.check_call(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, "/kaggle/working/repo"])
REPO = pathlib.Path("/kaggle/working/repo")
sys.path.insert(0, str(REPO))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
TEST_DIR = pathlib.Path(f"/kaggle/input/{DATASET_SLUG}/{TEST_SUBDIR}")
assert TEST_DIR.exists(), f"not found: {TEST_DIR}"
samples = sorted(p for p in TEST_DIR.iterdir() if p.is_dir())
print(f"{len(samples)} test samples")
print("first:", samples[0].name if samples else "")

In [ ]:
from src.submit import build_submission, zip_submission, _resolve_pipeline

render = _resolve_pipeline(PIPELINE)
OUT = pathlib.Path("/kaggle/working/submission")
stats = build_submission(TEST_DIR, OUT, render, quality=95, skip_existing=False)
print(stats)

In [ ]:
zip_path = zip_submission(OUT, "/kaggle/working/submission.zip")
print(zip_path, zip_path.stat().st_size // (1024 * 1024), "MB")